In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd

from langchain_ollama import ChatOllama
from langchain_anthropic import ChatAnthropic
from src.graphs.IdentificationGraph import build_identification_graph
from datetime import datetime


In [2]:
MAX_TRIALS = 10
model = ChatOllama(
    model="gemma4:31b-cloud",
    temperature=0,
    reasoning=False
)

# model = ChatAnthropic(
#     model="claude-sonnet-4-6",
#     temperature=0,
#     max_tokens=4096,
#     api_key=""
# )

# Optimizer focus for the optimization step. Allowed: OptimizationStrategy.META_PROMPTING,
# FEW_SHOT_PROMPTING, DECISION_RUBRIC, or NONE. Their string values ('meta_prompting',
# 'few_shot_prompting', 'decision_rubric', 'none') are also accepted; anything else
# raises a ValueError before any model call.
from src.models.OptimizationStrategy import OptimizationStrategy
STRATEGY = OptimizationStrategy.DECISION_RUBRIC


In [3]:
# 1. Define the Graph
descriptions = {
    "lib_desc": open("../inputs/descriptions/library.txt", encoding="utf-8").read(),
    "rental_desc": open("../inputs/descriptions/car_rental.txt", encoding="utf-8").read(),
    "ntss_desc": open("../inputs/descriptions/ntss.txt", encoding="utf-8").read(),
}

# 2. Load ground truths (upload library.csv, car_rental.csv, ntss.csv to
#    ground_truths/identification/ first - see that folder's README for the format)
try:
    truths = {
        "lib_truth": pd.read_csv("../ground_truths/identification/lib.csv"),
        "rent_truth": pd.read_csv("../ground_truths/identification/rental.csv"),
        "ntss_truth": pd.read_csv("../ground_truths/identification/ntss.csv"),
    }
except FileNotFoundError as e:
    raise FileNotFoundError(
        "Identification ground truths not found. Upload library.csv, car_rental.csv and "
        "ntss.csv to ground_truths/identification/ (columns: rule,phrase)"
    ) from e

workflow = build_identification_graph(
    model=model,
    descriptions=descriptions,
    truths=truths,
    max_trials=MAX_TRIALS
)
agent = workflow.compile()

# 3. Run the Optimizer
# Pulling initial prompt from the specified file
prompt_path = "../prompts/auto_prompt_evo/identification/t0/prompt.md"
try:
    with open(prompt_path, "r", encoding="utf-8") as f:
        initial_prompt = f.read()
    print("Successfully loaded prompt from:", prompt_path)
except FileNotFoundError:
    raise FileNotFoundError(f"Could not find file at {prompt_path}. Please check the path.")

if "<DESCRIPTION>" not in initial_prompt:
    raise ValueError("Initial prompt is missing the <DESCRIPTION> placeholder - "
                     "descriptions would never be injected!")

inputs = {
    "current_prompt": initial_prompt,
    "current_trial": 1,
    "optimization_strategy": STRATEGY,
    "lib_eval": {}, "rental_eval": {}, "ntss_eval": {}
}

print(f"\n[{datetime.now():%H:%M:%S}] Starting identification optimizer: {MAX_TRIALS} trials planned")
result = agent.invoke(inputs)
print(f"\n[{datetime.now():%H:%M:%S}] Optimizer finished after {result['current_trial'] - 1} completed trials")


Successfully loaded prompt from: ../prompts/auto_prompt_evo/identification/t0/prompt.md

[10:57:21] Starting identification optimizer: 10 trials planned
[10:57:21] Trial 1 | lib: identifying domain phrases...
[10:57:22] Trial 1 | lib: done (640 chars)
[10:57:22] Trial 1 | rental: identifying domain phrases...
[10:57:23] Trial 1 | rental: done (1392 chars)
[10:57:23] Trial 1 | ntss: identifying domain phrases...
[10:57:27] Trial 1 | ntss: done (1276 chars)

--- Identification Evaluation Results (trial 1) ---
Lib: p-0.78 r-0.56 f1-0.65 (TP=31 FP=9 FN=24)
Rent: p-0.34 r-0.44 f1-0.38 (TP=27 FP=53 FN=35)
Ntss: p-0.63 r-0.55 f1-0.59 (TP=48 FP=28 FN=39)
[10:57:27] Trial 1 | score: wrote scorecard to ../prompts/auto_prompt_evo/identification/t1/scorecard.md
[10:57:27] Trial 1 | optimize: strategy 'decision_rubric'; asking model to improve the prompt...
[10:57:31] Trial 1 | optimize: done
[10:57:31] Trial 1 | checkpoint: saving optimized prompt to ../prompts/auto_prompt_evo/identification/t1/pr